# **가계신용대출 금리 트렌드 — 통합 노트북**

> 작성: 2025년 5월 기준 통합본
>
> 기존 두 파일(`가계신용대출 금리 트랜드_DB.ipynb` 크롤링 + `가계신용대출 금리 트랜드 코드집.ipynb` 분석)을 하나로 합친 단일 파일.
>
> ## 흐름
>
> ```
> [수집]  협회 사이트 크롤링  →  RAW CSV 백업 + 메모리 DataFrame
> [전처리] 코드화 / melt / concat / 주요기관 추출
> [시각화] Plotly 인터랙티브 그래프 (int_step / cs_viline)
> [출력]  단일 HTML 보고서 (반응형 레이아웃)
> ```
>
> ## 진행 상태
>
> | 단계 | 상태 |
> |---|---|
> | M0. 골격 | ✅ |
> | M1. 신용점수 금리 6업권 (9구간 확장) | ✅ 1차 |
> | M2. 잔액·중금리·금리대 신규 크롤링 | ⬜ TODO |
> | M3. 메모리 전처리 | ✅ 신용점수 금리만 |
> | M4. int_step_graph → Plotly | ✅ |
> | M5. cs_viline_graph → Plotly | ⬜ TODO |
> | M6. HTML 보고서 출력 | ✅ 1차 |

## 0. 사용자 입력 / 환경설정

### 0.1. 수집 대상 연월

In [ ]:
# ────────── 사용자 입력 ──────────
# 단일 월: target_months = [(2024, 10)]
# 다월 백필: target_months = [(y, m) for y in [2024] for m in range(1, 11)]
target_months = [(2024, 10)]    # ← 여기만 바꾸면 됨
# ─────────────────────────────────

# 시각화/보고서 기준 월 (target_months의 마지막 월)
year, month = target_months[-1]
YM = f"{year}{month:02d}"

# 출력 폴더 (현재 노트북 기준)
import os
OUT_RAW = "./RAW"                # 월별 하위 폴더에 분리 저장 (RAW_YYYYMM)
OUT_REPORT = "./report"
os.makedirs(OUT_RAW, exist_ok=True)
os.makedirs(OUT_REPORT, exist_ok=True)

print(f"수집 대상: {len(target_months)}개월  ({target_months[0]} ~ {target_months[-1]})")
print(f"기준 월 (시각화): {year}년 {month}월  (YM={YM})")
print(f"RAW 저장: {OUT_RAW}/RAW_YYYYMM/")
print(f"보고서:   {OUT_REPORT}")


### 0.2. 라이브러리

In [2]:
# 표준
import os, re, io, json, ssl, warnings
import urllib3
from urllib.parse import urlencode, unquote
from datetime import datetime

# 데이터
import numpy as np
import pandas as pd
from pandas import DataFrame

# 크롤링
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.ssl_ import create_urllib3_context
from bs4 import BeautifulSoup

# 시각화 (Plotly)
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

print('OK')

ModuleNotFoundError: No module named 'bs4'

### 0.3. 공통 유틸

In [ ]:
# ───── TLS Cipher Suite 강제 (협회 사이트 호환) ─────
class TLSAdapter(HTTPAdapter):
    def init_poolmanager(self, *args, **kwargs):
        ctx = ssl.create_default_context()
        ctx.set_ciphers("AES128-SHA256")
        kwargs["ssl_context"] = ctx
        return super().init_poolmanager(*args, **kwargs)

# ───── 공시연월 → 실제 데이터 연월 (1개월 차감) ─────
def subtract_month(df, col_name='공시연월', new_col_name='연월'):
    df[new_col_name] = pd.to_datetime(df[col_name].astype(str), format='%Y%m') - pd.DateOffset(months=1)
    df[new_col_name] = df[new_col_name].dt.strftime('%Y%m').astype(str)
    return df

# ───── % → 실수 (코드집의 percent_to_real) ─────
def percent_to_real(df, cols):
    for c in cols:
        df[c] = pd.to_numeric(df[c], errors='coerce') / 100
    return df

# ───── 빈 행 이하 절단 ─────
def erase_null_row(df):
    first_empty_index = df[df.isnull().all(axis=1)].index
    if not first_empty_index.empty:
        df = df[:first_empty_index[0]]
    return df

# ───── % 표기 헬퍼 ─────
def pct1(v):
    return f"{v*100:.1f}%" if pd.notna(v) else ""
def pct2(v):
    return f"{v*100:.2f}%" if pd.notna(v) else ""

print('utils OK')

### 0.4. 코드 매핑 / 사명 리스트

* `cs_code` : 신용점수 구간 → 3자리 코드 (코드집 기준 9구간)
* `cs_code_easy` : 9구간 → 단순화된 등급 라벨 (시각화 색상용)
* `cs_code_easy_rev` : 컬러바 라벨
* `int_code` : 금리구간 → 코드 (잔여 단계에서 사용)

In [ ]:
# 신용점수 9구간 (코드집 기준)
cs_code = {
    '1000~951점':'011', '950~901점':'012',
    '900~851점':'021', '850~801점':'022',
    '800~751점':'031', '750~701점':'032',
    '700~651점':'041', '650~601점':'042',
    '600점이하':'050',
    # _DB의 4구간(900대/800대/700대/600대)을 9구간 코드로 매핑
    '900점대':'010', '800점대':'020', '700점대':'030', '600점대':'040',
    # 은행권 일부 구코드
    '900점 초과':'010', '801점~900점':'020', '701점~800점':'030',
    '601점~700점':'040', '501점~600점':'050',
    '401점~500점':'060', '301점~400점':'070', '300점 이하':'080',
}
cs_code_rev = {v: k for k, v in cs_code.items()}

cs_code_easy = {
    '011':'9','012':'9','010':'9',
    '021':'8','022':'8','020':'8',
    '031':'7','032':'7','030':'7',
    '041':'6','042':'6','040':'6',
    '050':'5','060':'4','070':'3','080':'3L'
}
cs_code_easy_rev = {
    '1':'900점대','2':'800점대','3':'700점대','4':'600점대',
    '5':'500점대','6':'400점대','7':'300점대','8':'300점이하'
}

# 금리구간 (M2에서 사용)
int_code = {
    '4%미만':'010','4~5%미만':'012','5~6%미만':'013','6~7%미만':'014',
    '7~8%미만':'015','8~9%미만':'016','9~10%미만':'017','10%이상':'018',
    '10%미만':'017','10~12%':'018','12~14%':'019','14~16%':'020',
    '16~18%':'021','18~20%':'022',
    '10%이하':'017','12%이하':'018','14%이하':'019','16%이하':'020',
    '18%이하':'021','20%이하':'022',
}

# 사명 리스트 (업권 분류용)
tot_nationwide = ['KB국민은행','우리은행','SC제일은행','신한은행','하나은행',
                  '한국씨티은행','NH농협은행','IBK기업은행']
tot_local = ['iM뱅크(구 대구은행)','BNK부산은행','광주은행','제주은행',
             '전북은행','BNK경남은행']
tot_internet = ['케이뱅크','카카오뱅크','토스뱅크']
tot_special = ['Sh수협은행']

def categorize_bank(name):
    if name in tot_nationwide: return '시중은행'
    if name in tot_internet:   return '인터넷은행'
    if name in tot_local:      return '지방은행'
    if name in tot_special:    return '특수은행'
    # 키워드 매칭 fallback
    if any(k in name for k in ['대구','부산','광주','제주','전북','경남']):
        return '지방은행'
    if any(k in name for k in ['케이','카카오','토스']):
        return '인터넷은행'
    if any(k in name for k in ['수협','기업']):
        return '특수은행'
    return '시중은행'

print('codes OK')

## 1. 데이터 수집 (크롤링)

각 함수는 **(연월, 평균금리, 신용점수 구간별 금리)** 를 메모리 DataFrame으로 반환하고, **동시에 RAW CSV로도 저장**합니다.

**다월 백필 지원**: 셀 0.1의 `target_months` 리스트에 (년, 월) 튜플을 여러 개 넣으면 자동으로 누적 수집한다.

공통 출력 스키마:

```
사명 | 업권 | 상품구분 | 연월 | 평균금리 | 신용점수구간 | 구간평균금리 | (CB기관) | (평균신용점수)
```

신용점수 구간이 4구간만 제공되는 사이트(저축/카드/캐피탈)는 `010/020/030/040` 코드로,
9구간 제공되는 사이트(은행)는 `011/012/021/022/...` 코드로 저장됩니다.

### 1.1. 저축은행 — 신용점수별 금리 (저축은행중앙회)

In [ ]:
def crawl_savings_cs(year, month):
    url = "https://www.fsb.or.kr/ratloanconf_0200.jct"
    payload = {"_JSON_": json.dumps({
        "SORT_COLUMN":"","SORT":"","PRE_MONTH_MONEY":"",
        "SUBMIT_MONTH": f"{year:04d}{month:02d}"
    })}
    r = requests.post(url, data=payload, timeout=30)
    r.raise_for_status()
    rec = json.loads(r.text)['REC']
    df = pd.DataFrame(rec)
    df = df[['BANK_NAME','SUBMIT_MONTH','A_RATE1_3','A_RATE1','A_RATE2','A_RATE3','A_RATE_AVE']]
    df.columns = ['사명','공시연월','900점대','800점대','700점대','600점대','평균금리']
    df['업권'] = '저축은행'
    df['상품구분'] = '일반신용'
    df = subtract_month(df, '공시연월', '연월')

    # wide → long
    long = df.melt(
        id_vars=['사명','업권','상품구분','연월','평균금리'],
        value_vars=['900점대','800점대','700점대','600점대'],
        var_name='신용점수구간', value_name='구간평균금리'
    )
    # % → 실수 (저축은행중앙회 응답이 % 값인지 확인 필요)
    long['평균금리'] = pd.to_numeric(long['평균금리'], errors='coerce') / 100
    long['구간평균금리'] = pd.to_numeric(long['구간평균금리'], errors='coerce') / 100
    # 코드화
    long['신용점수구간'] = long['신용점수구간'].map(cs_code)
    return long


# 다월 백필: target_months 전체 순회
_savings_frames = []
for _y, _m in target_months:
    _ym = f"{_y}{_m:02d}"
    _out_dir = f"{OUT_RAW}/RAW_{_ym}"
    os.makedirs(_out_dir, exist_ok=True)
    try:
        _df = crawl_savings_cs(_y, _m)
        _df.to_csv(f"{_out_dir}/savings_cs.csv", index=False, encoding='utf-8-sig')
        _savings_frames.append(_df)
        print(f"  [{_ym}] savings_cs OK — {len(_df)} rows")
    except Exception as e:
        print(f"  [{_ym}] savings_cs FAILED: {type(e).__name__}: {e}")
savings_cs_df = pd.concat(_savings_frames, ignore_index=True) if _savings_frames else pd.DataFrame()
print(f"savings_cs (전체): {len(savings_cs_df):,} rows, 월: {sorted(savings_cs_df['연월'].unique().tolist()) if not savings_cs_df.empty else []}")


### 1.2. 신용카드 — 카드론 / 현금서비스 / 일반신용 (여신금융협회)

In [ ]:
# 카드사 mcSeq (여신금융협회 내부 ID)
CARD_MC_SEQ = [31, 96, 1, 106, 14, 13, 12, 98, 502, 108, 619, 11, 97, 105, 103, 22]

def _crefia_card_cs(year, month, cgc_mode, list_seq, point_cols, label):
    """여신금융협회 카드 신용점수별 금리 공통 함수
    cgc_mode: 25(카드론) / 20(현금서비스)
    point_cols: ('cgCardPoint1', 'cgCardPoint2', ...) 또는 ('cgMoneyPoint1', ...)
    """
    base = "https://gongsi.crefia.or.kr/portal/creditcard/creditcardDisclosureDetail{}Ajax".format(cgc_mode)
    with requests.session() as s:
        s.mount("https://", TLSAdapter())
        # 1단계: cgcSeq 찾기
        r1 = s.get(base, params={"cgcSeq": list_seq, "cgcMode": cgc_mode, "cgcYyyy": year, "mcSeq": []}, timeout=30)
        r1.raise_for_status()
        cgc_seq = None
        for item in json.loads(r1.text)['configListMm']:
            if item['cgcquarter'] == month:
                cgc_seq = item['cgcSeq']; break
        if cgc_seq is None:
            raise ValueError(f"{label}: {year}-{month}에 해당하는 cgcSeq 없음")
        # 2단계: 데이터
        r2 = s.get(base, params={"cgcSeq": cgc_seq, "cgcMode": cgc_mode, "cgcYyyy": year, "mcSeq": CARD_MC_SEQ}, timeout=30)
        r2.raise_for_status()
    df = pd.DataFrame(json.loads(r2.text)['resultList'])
    df = df[['mcCompany','cgcSeq', *point_cols]]
    df.columns = ['사명','공시연월','900점대','800점대','700점대','600점대','평균금리']
    df['공시연월'] = f"{year}{month:02d}"
    df['업권'] = '신용카드'
    df['상품구분'] = label
    df = subtract_month(df, '공시연월', '연월')
    long = df.melt(
        id_vars=['사명','업권','상품구분','연월','평균금리'],
        value_vars=['900점대','800점대','700점대','600점대'],
        var_name='신용점수구간', value_name='구간평균금리'
    )
    long['평균금리'] = pd.to_numeric(long['평균금리'], errors='coerce') / 100
    long['구간평균금리'] = pd.to_numeric(long['구간평균금리'], errors='coerce') / 100
    long['신용점수구간'] = long['신용점수구간'].map(cs_code)
    return long

def crawl_card_loan_cs(year, month):
    return _crefia_card_cs(year, month, 25, 1458,
        ('cgCardPoint1','cgCardPoint2','cgCardPoint3','cgCardPoint4','cgCardPointAvg'), '카드론')

def crawl_card_cash_cs(year, month):
    return _crefia_card_cs(year, month, 20, 1460,
        ('cgMoneyPoint1','cgMoneyPoint2','cgMoneyPoint3','cgMoneyPoint4','cgMoneyPointAvg'), '현금서비스')

def crawl_card_normal_cs(year, month):
    """카드사 일반신용대출 — clgcMode=11, cardItem=카드사ID"""
    url = "https://gongsi.crefia.or.kr/portal/creditloan/creditloanDisclosureDetail11/ajax"
    card_item = "14,13,12,502,619,103"
    with requests.session() as s:
        s.mount("https://", TLSAdapter())
        r1 = s.get(url, params={"clgcMode":11,"cardItem":card_item,"clgcSeq":521,"clgcYyyy":year}, timeout=30)
        r1.raise_for_status()
        clgc_seq = None
        for item in json.loads(r1.text)['configList']:
            if (item['clgcquarter']==month) and (item['clgcYear']==str(year)):
                clgc_seq = item['clgcSeq']; break
        if clgc_seq is None:
            raise ValueError(f"card_normal: {year}-{month}에 해당하는 clgcSeq 없음")
        r2 = s.get(url, params={"clgcMode":11,"cardItem":card_item,"clgcSeq":clgc_seq,"clgcYyyy":year}, timeout=30)
        r2.raise_for_status()
    df = pd.DataFrame(json.loads(r2.text)['resultList'])
    df = df[['mcCompany','clgcSeq','clgPoint1','clgPoint2','clgPoint3','clgPoint4','clgPointAvg']]
    df.columns = ['사명','공시연월','900점대','800점대','700점대','600점대','평균금리']
    df['공시연월'] = f"{year}{month:02d}"
    df['업권'] = '신용카드'
    df['상품구분'] = '일반신용'
    df = subtract_month(df, '공시연월', '연월')
    long = df.melt(
        id_vars=['사명','업권','상품구분','연월','평균금리'],
        value_vars=['900점대','800점대','700점대','600점대'],
        var_name='신용점수구간', value_name='구간평균금리'
    )
    long['평균금리'] = pd.to_numeric(long['평균금리'], errors='coerce') / 100
    long['구간평균금리'] = pd.to_numeric(long['구간평균금리'], errors='coerce') / 100
    long['신용점수구간'] = long['신용점수구간'].map(cs_code)
    return long


# 다월 백필
_card_frames = {'card_loan_cs': [], 'card_cash_cs': [], 'card_normal_cs': []}
for _y, _m in target_months:
    _ym = f"{_y}{_m:02d}"
    _out_dir = f"{OUT_RAW}/RAW_{_ym}"
    os.makedirs(_out_dir, exist_ok=True)
    for _name, _fn in [('card_loan_cs', crawl_card_loan_cs),
                         ('card_cash_cs', crawl_card_cash_cs),
                         ('card_normal_cs', crawl_card_normal_cs)]:
        try:
            _df = _fn(_y, _m)
            _df.to_csv(f"{_out_dir}/{_name}.csv", index=False, encoding='utf-8-sig')
            _card_frames[_name].append(_df)
            print(f"  [{_ym}] {_name} OK — {len(_df)} rows")
        except Exception as e:
            print(f"  [{_ym}] {_name} FAILED: {type(e).__name__}: {e}")
card_loan_cs_df = pd.concat(_card_frames['card_loan_cs'], ignore_index=True) if _card_frames['card_loan_cs'] else pd.DataFrame()
card_cash_cs_df = pd.concat(_card_frames['card_cash_cs'], ignore_index=True) if _card_frames['card_cash_cs'] else pd.DataFrame()
card_normal_cs_df = pd.concat(_card_frames['card_normal_cs'], ignore_index=True) if _card_frames['card_normal_cs'] else pd.DataFrame()
print(f"card (전체): loan={len(card_loan_cs_df):,} / cash={len(card_cash_cs_df):,} / normal={len(card_normal_cs_df):,}")


### 1.3. 캐피탈 — 신용점수별 금리 (여신금융협회)

In [ ]:
def crawl_capital_cs(year, month):
    url = "https://gongsi.crefia.or.kr/portal/creditloan/creditloanDisclosureDetail11/ajax"
    card_item = "134,39,40,623,130,41,25,156,6,55,32,58,52,61,57,64"
    with requests.session() as s:
        s.mount("https://", TLSAdapter())
        r1 = s.get(url, params={"clgcMode":11,"cardItem":card_item,"clgcSeq":521,"clgcYyyy":year}, timeout=30)
        r1.raise_for_status()
        clgc_seq = None
        for item in json.loads(r1.text)['configList']:
            if (item['clgcquarter']==month) and (item['clgcYear']==str(year)):
                clgc_seq = item['clgcSeq']; break
        if clgc_seq is None:
            raise ValueError(f"capital_cs: {year}-{month}에 해당하는 clgcSeq 없음")
        r2 = s.get(url, params={"clgcMode":11,"cardItem":card_item,"clgcSeq":clgc_seq,"clgcYyyy":year}, timeout=30)
        r2.raise_for_status()
    df = pd.DataFrame(json.loads(r2.text)['resultList'])
    df = df[['mcCompany','clgcSeq','clgPoint1','clgPoint2','clgPoint3','clgPoint4','clgPointAvg']]
    df.columns = ['사명','공시연월','900점대','800점대','700점대','600점대','평균금리']
    df['공시연월'] = f"{year}{month:02d}"
    df['업권'] = '캐피탈'
    df['상품구분'] = '일반신용'
    df = subtract_month(df, '공시연월', '연월')
    long = df.melt(
        id_vars=['사명','업권','상품구분','연월','평균금리'],
        value_vars=['900점대','800점대','700점대','600점대'],
        var_name='신용점수구간', value_name='구간평균금리'
    )
    long['평균금리'] = pd.to_numeric(long['평균금리'], errors='coerce') / 100
    long['구간평균금리'] = pd.to_numeric(long['구간평균금리'], errors='coerce') / 100
    long['신용점수구간'] = long['신용점수구간'].map(cs_code)
    return long


# 다월 백필
_capital_frames = []
for _y, _m in target_months:
    _ym = f"{_y}{_m:02d}"
    _out_dir = f"{OUT_RAW}/RAW_{_ym}"
    os.makedirs(_out_dir, exist_ok=True)
    try:
        _df = crawl_capital_cs(_y, _m)
        _df.to_csv(f"{_out_dir}/capital_cs.csv", index=False, encoding='utf-8-sig')
        _capital_frames.append(_df)
        print(f"  [{_ym}] capital_cs OK — {len(_df)} rows")
    except Exception as e:
        print(f"  [{_ym}] capital_cs FAILED: {type(e).__name__}: {e}")
capital_cs_df = pd.concat(_capital_frames, ignore_index=True) if _capital_frames else pd.DataFrame()
print(f"capital_cs (전체): {len(capital_cs_df):,} rows")


### 1.4. 은행 — 신용점수별 금리 (은행연합회)

시점별 URL 4단계 분기 + 9구간 ↔ 4구간 응답 분기.

In [ ]:
# 은행 19개 (URL 인코딩 EUC-KR) — _DB의 str payload 그대로
BANK_STR = ("KDB%BB%EA%BE%F7%C0%BA%C7%E0|NH%B3%F3%C7%F9%C0%BA%C7%E0|%BD%C5%C7%D1%C0%BA%C7%E0|%BF%EC%B8%AE%C0%BA%C7%E0|"
            "SC%C1%A6%C0%CF%C0%BA%C7%E0|%C7%CF%B3%AA%C0%BA%C7%E0|IBK%B1%E2%BE%F7%C0%BA%C7%E0|KB%B1%B9%B9%CE%C0%BA%C7%E0|"
            "%C7%D1%B1%B9%BE%BE%C6%BC%C0%BA%C7%E0|Sh%BC%F6%C7%F9%C0%BA%C7%E0|iM%B9%F0%C5%A9%28%B1%B8+%B4%EB%B1%B8%C0%BA%C7%E0%29|"
            "BNK%BA%CE%BB%EA%C0%BA%C7%E0|%B1%A4%C1%D6%C0%BA%C7%E0|%C1%A6%C1%D6%C0%BA%C7%E0|%C0%FC%BA%CF%C0%BA%C7%E0|"
            "BNK%B0%E6%B3%B2%C0%BA%C7%E0|%C4%C9%C0%CC%B9%F0%C5%A9|%C4%AB%C4%AB%BF%C0%B9%F0%C5%A9|%C5%E4%BD%BA%B9%F0%C5%A9")

def _bank_url(ym):
    if ym > '202306':
        return ("https://portal.kfb.or.kr/compare/loan_household_new_search_result_new.php",
                "https://portal.kfb.or.kr/compare/loan_household_new.php")
    if ym > '202207':
        return ("https://portal.kfb.or.kr/compare/loan_household_new_search_result_lys.php",
                "https://portal.kfb.or.kr/compare/loan_household_new.php")
    if ym > '202101':
        return ("https://portal.kfb.or.kr/compare/loan_household_new_search_result.php",
                "https://portal.kfb.or.kr/compare/loan_household_new_202202.php")
    return ("https://portal.kfb.or.kr/compare/loan_household_search_result.php",
            "https://portal.kfb.or.kr/compare/loan_household.php")

def crawl_bank_cs(year, month):
    ym = f"{year}{month:02d}"
    url, referer = _bank_url(ym)
    payload = {"year":year, "month":f"{month:02d}", "opt_1":3, "detail":0, "str":BANK_STR, "select_new_balance":1}
    headers = {
        "Accept":"*/*",
        "Content-Type":"application/x-www-form-urlencoded; charset=UTF-8",
        "Origin":"https://portal.kfb.or.kr",
        "Referer":referer,
        "User-Agent":"Mozilla/5.0",
        "X-Requested-With":"XMLHttpRequest",
    }
    with requests.session() as s:
        s.mount("https://", TLSAdapter())
        r = s.post(url, data=payload, headers=headers, timeout=30)
        r.raise_for_status()
        r.encoding = 'euc-kr'
    soup = BeautifulSoup(r.text, 'html.parser')
    table = soup.find('table')
    if table is None:
        raise ValueError('은행연합회 응답에 table 없음')
    rows = table.find_all('tr')[2:]
    data = []
    for row in rows:
        cols = [c.text.strip() for c in row.find_all('td')]
        if len(cols) < 3: continue
        cols[1] = cols[1] + cols[2]   # 사명 두 칼럼 합치기
        data.append(cols)
    df = pd.DataFrame(data)
    if df.empty: raise ValueError('파싱 결과 비어있음')

    # 9구간 시기 (>202207): 컬럼 인덱스 2~9가 9구간(2개씩 묶이지 않은 raw)
    if ym > '202207':
        # 컬럼: 0=사명, 2~9=신용점수 9구간(둘씩 묶기 전), 11=평균금리
        df = df[[0,2,3,4,5,6,7,8,9,11]].copy()
        for col in [2,3,4,5,6,7,8,9,11]:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        df.columns = ['사명','1000~951점','950~901점','900~851점','850~801점',
                       '800~751점','750~701점','700~651점','650~601점','평균금리']
        df['업권'] = df['사명'].apply(categorize_bank)
        df['상품구분'] = '일반신용'
        df['공시연월'] = ym
        df = subtract_month(df, '공시연월', '연월')
        df.dropna(subset=['평균금리'], inplace=True)
        long = df.melt(
            id_vars=['사명','업권','상품구분','연월','평균금리'],
            value_vars=['1000~951점','950~901점','900~851점','850~801점',
                         '800~751점','750~701점','700~651점','650~601점'],
            var_name='신용점수구간', value_name='구간평균금리'
        )
    else:
        df = df[[0,2,3,4,5,7]].copy()
        df.columns = ['사명','900점대','800점대','700점대','600점대','평균금리']
        for col in ['900점대','800점대','700점대','600점대','평균금리']:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        df['업권'] = df['사명'].apply(categorize_bank)
        df['상품구분'] = '일반신용'
        df['공시연월'] = ym
        df = subtract_month(df, '공시연월', '연월')
        df.dropna(subset=['평균금리'], inplace=True)
        long = df.melt(
            id_vars=['사명','업권','상품구분','연월','평균금리'],
            value_vars=['900점대','800점대','700점대','600점대'],
            var_name='신용점수구간', value_name='구간평균금리'
        )
    long['평균금리'] = long['평균금리'] / 100
    long['구간평균금리'] = long['구간평균금리'] / 100
    long['신용점수구간'] = long['신용점수구간'].map(cs_code)
    return long


# 다월 백필
_bank_frames = []
for _y, _m in target_months:
    _ym = f"{_y}{_m:02d}"
    _out_dir = f"{OUT_RAW}/RAW_{_ym}"
    os.makedirs(_out_dir, exist_ok=True)
    try:
        _df = crawl_bank_cs(_y, _m)
        _df.to_csv(f"{_out_dir}/bank_cs.csv", index=False, encoding='utf-8-sig')
        _bank_frames.append(_df)
        print(f"  [{_ym}] bank_cs OK — {len(_df)} rows  (업권: {sorted(_df['업권'].unique())})")
    except Exception as e:
        print(f"  [{_ym}] bank_cs FAILED: {type(e).__name__}: {e}")
bank_cs_df = pd.concat(_bank_frames, ignore_index=True) if _bank_frames else pd.DataFrame()
print(f"bank_cs (전체): {len(bank_cs_df):,} rows")


### 1.5. 저축은행 금리대별 취급비중 (savings_int)

저축은행중앙회 — `ratloanconf_0300.jct` 엔드포인트.

출력 long 포맷: `사명 | 업권 | 상품구분 | 연월 | 금리구간 | 취급비중`

> M2 미구현 항목 (P2~P5): 은행·카드 금리대별 / 잔액 / 중금리. 각 사이트별 엔드포인트 분석 후 추가 예정.


In [ ]:
def crawl_savings_int(year, month):
    """저축은행 금리대별 취급비중 (저축은행중앙회).

    Returns: long-format DataFrame
        ['사명','업권','상품구분','연월','금리구간','취급비중']
    """
    url = "https://www.fsb.or.kr/ratloanconf_0300.jct"
    payload = {"SORT":"", "SUBMIT_MONTH": f"{year}{month:02d}"}

    s = requests.Session()
    s.mount("https://", TLSAdapter())
    r = s.post(url, data={"_JSON_": json.dumps(payload)}, verify=False, timeout=30)
    r.raise_for_status()
    data = json.loads(r.text)
    df = pd.DataFrame(data["REC"])

    cols = ["BANK_NAME","HANDING_WEIGHT_10","HANDING_WEIGHT_12","HANDING_WEIGHT_14",
            "HANDING_WEIGHT_16","HANDING_WEIGHT_18","HANDING_WEIGHT_20"]
    df = df[cols]
    df.columns = ['사명','10%이하','12%이하','14%이하','16%이하','18%이하','20%이하']

    df['업권'] = '저축은행'
    df['상품구분'] = '일반신용'
    df['연월'] = f"{year}{month:02d}"

    # % → 실수
    weight_cols = ['10%이하','12%이하','14%이하','16%이하','18%이하','20%이하']
    for c in weight_cols:
        df[c] = pd.to_numeric(df[c], errors='coerce') / 100

    # wide → long
    long = df.melt(
        id_vars=['사명','업권','상품구분','연월'],
        value_vars=weight_cols,
        var_name='금리구간', value_name='취급비중'
    )
    long = long.dropna(subset=['취급비중'])
    return long


# 다월 백필
_savings_int_frames = []
for _y, _m in target_months:
    _ym = f"{_y}{_m:02d}"
    _out_dir = f"{OUT_RAW}/RAW_{_ym}"
    os.makedirs(_out_dir, exist_ok=True)
    try:
        _df = crawl_savings_int(_y, _m)
        _df.to_csv(f"{_out_dir}/savings_int.csv", index=False, encoding='utf-8-sig')
        _savings_int_frames.append(_df)
        print(f"  [{_ym}] savings_int OK — {len(_df)} rows")
    except Exception as e:
        print(f"  [{_ym}] savings_int FAILED: {type(e).__name__}: {e}")
savings_int_df = pd.concat(_savings_int_frames, ignore_index=True) if _savings_int_frames else pd.DataFrame()
print(f"savings_int (전체): {len(savings_int_df):,} rows")


## 2. 메모리 기반 전처리

앞 단계에서 크롤링된 모든 신용점수별 금리 DataFrame을 합쳐 통합 테이블 `tot_cs`를 만든다.

In [ ]:
frames = [df for df in [savings_cs_df, card_loan_cs_df, card_cash_cs_df,
                         card_normal_cs_df, capital_cs_df, bank_cs_df] if not df.empty]
if not frames:
    raise RuntimeError('수집된 데이터가 하나도 없습니다. 셀 1.x를 다시 실행하세요.')

tot_cs = pd.concat(frames, ignore_index=True)
tot_cs = tot_cs.dropna(subset=['평균금리'])
print(f"tot_cs: {len(tot_cs):,} rows  | 업권: {sorted(tot_cs['업권'].unique())}")
tot_cs.head()

# ───── 금리대별 취급비중 통합 (M2 P1: 현재 저축은행만) ─────
int_frames = [df for df in [savings_int_df] if not df.empty]
if int_frames:
    tot_int = pd.concat(int_frames, ignore_index=True)
    # 금리구간 코드화
    tot_int['금리구간'] = tot_int['금리구간'].map(int_code)
    print(f"tot_int: {len(tot_int):,} rows  | 월: {sorted(tot_int['연월'].unique())}  | 업권: {sorted(tot_int['업권'].unique())}")
else:
    tot_int = pd.DataFrame()
    print('tot_int: 비어있음 (M2 P1 데이터 없음)')
tot_int.head()


### 2.1. 주요기관 추출 (잔액 미수집 단계 — 사명 리스트 기반 임시 필터)

M2에서 잔액 크롤링이 들어오면 코드집과 동일하게 잔액 임계값 필터로 교체.

In [ ]:
# 미리보기 단계: 잔액이 없으니 사명 리스트로 주요기관 임시 정의
major_nationwide_tmp = ['KB국민은행','우리은행','신한은행','하나은행','NH농협은행','IBK기업은행']
major_local_tmp     = ['iM뱅크(구 대구은행)','BNK부산은행','광주은행','BNK경남은행']
major_internet_tmp  = ['케이뱅크','카카오뱅크','토스뱅크']
major_card_tmp      = ['삼성카드','신한카드','KB국민카드','현대카드','롯데카드','하나카드','우리카드','비씨카드']
major_capital_tmp   = ['현대캐피탈','KB캐피탈','하나캐피탈','우리금융캐피탈','메리츠캐피탈']
major_savings_tmp   = ['SBI','OK','웰컴','한국투자','애큐온','페퍼','다올','상상인플러스']

def _filter(df, ind, names):
    return df[(df['업권']==ind) & (df['사명'].isin(names))]

tot_cs_major = pd.concat([
    _filter(tot_cs,'시중은행',major_nationwide_tmp),
    _filter(tot_cs,'인터넷은행',major_internet_tmp),
    _filter(tot_cs,'지방은행',major_local_tmp),
    _filter(tot_cs,'신용카드',major_card_tmp),
    _filter(tot_cs,'캐피탈',major_capital_tmp),
    _filter(tot_cs,'저축은행',major_savings_tmp),
], ignore_index=True)

print(f"tot_cs_major: {len(tot_cs_major):,} rows")
tot_cs_major.groupby('업권').size()

## 3. 시각화 (Plotly)

### 3.1. int_step_graph (Plotly 버전)

원본 matplotlib 클래스의 핵심 시각 정체성을 유지:

* **가변 x축 간격** — `start_month` 이전은 0.2, 이후는 1 단위 누적
* **계단형 라인** (`shape='hv'`)
* **3계층 중첩** — 업권 평균(굵게) / 개별사(얇게, mode='on') / 기준사(오렌지, mode_standard='on')
* **호버에 사명·연월·금리 표시** (Plotly 인터랙션)

현재 미리보기는 **시계열이 1개월치(이번 달 단월)만 있어서** step 차트가 의미가 작음. 누적 백필 시 진가 발휘.
M3 본단계에서는 과거 데이터를 누적 로드하여 다월 트렌드 표시 예정.

In [ ]:
INDIV_COLOR = {
    '시중은행':'#1f1f1f','인터넷은행':'#666','지방은행':'#aaa',
    '신용카드':'#3070c0','캐피탈':'#2ca02c','저축은행':'#d62728',
    '특수은행':'#9467bd'
}

def plot_int_step(df, industry_list, standard_name=None, start_month=None,
                   show_individual=False, title=''):
    """코드집 int_step_graph의 Plotly 1차 이식.
    df: tot_cs_major 형식 (사명/업권/상품구분/연월/평균금리/...)
    industry_list: 표시할 업권 리스트
    standard_name: 강조할 기준사 (예: 'OK')
    start_month: 'YYYYMM' 이전은 x축 간격 좁힘
    """
    # 평균금리 단월 집계 (사명 단위)
    base = (df.groupby(['사명','업권','상품구분','연월'], as_index=False)['평균금리']
              .mean())
    months_sorted = sorted(base['연월'].unique())
    if start_month is None:
        start_month = months_sorted[0]

    # 가변 x축 위치 매핑
    x_pos_map, prev = {}, 0
    for m in months_sorted:
        prev += 0.2 if m <= start_month else 1
        x_pos_map[m] = prev

    base['x_pos'] = base['연월'].map(x_pos_map)

    fig = go.Figure()

    # 업권 평균 라인 (굵게)
    ind = (base[base['업권'].isin(industry_list)]
             .groupby(['업권','연월'], as_index=False)['평균금리'].mean())
    ind['x_pos'] = ind['연월'].map(x_pos_map)
    for industry, sub in ind.groupby('업권', sort=False):
        sub = sub.sort_values('연월')
        fig.add_trace(go.Scatter(
            x=sub['x_pos'], y=sub['평균금리'],
            mode='lines+markers',
            line=dict(shape='hv', width=3, color=INDIV_COLOR.get(industry,'#333')),
            marker=dict(size=6),
            name=f"{industry} 평균",
            hovertemplate=f"<b>{industry}</b><br>%{{customdata}}<br>금리=%{{y:.2%}}<extra></extra>",
            customdata=sub['연월']
        ))

    # 개별사 (얇게)
    if show_individual:
        for (ind_, comp), sub in base[base['업권'].isin(industry_list)].groupby(['업권','사명'], sort=False):
            sub = sub.sort_values('연월')
            fig.add_trace(go.Scatter(
                x=sub['x_pos'], y=sub['평균금리'],
                mode='lines',
                line=dict(shape='hv', width=1, color=INDIV_COLOR.get(ind_,'#999'), dash='dot'),
                opacity=0.5,
                name=comp,
                hovertemplate=f"<b>{comp}</b> ({ind_})<br>%{{customdata}}<br>금리=%{{y:.2%}}<extra></extra>",
                customdata=sub['연월']
            ))

    # 기준사 강조
    if standard_name is not None:
        std = base[base['사명']==standard_name].sort_values('연월')
        if not std.empty:
            fig.add_trace(go.Scatter(
                x=std['x_pos'], y=std['평균금리'],
                mode='lines+markers',
                line=dict(shape='hv', width=3, color='darkorange'),
                marker=dict(size=8, symbol='diamond'),
                name=f"{standard_name} (기준)",
                hovertemplate=f"<b>{standard_name}</b><br>%{{customdata}}<br>금리=%{{y:.2%}}<extra></extra>",
                customdata=std['연월']
            ))

    fig.update_layout(
        title=dict(text=title, font=dict(size=14)),
        yaxis=dict(tickformat='.1%', title='평균금리'),
        xaxis=dict(
            title='연월',
            tickmode='array',
            tickvals=[x_pos_map[m] for m in months_sorted],
            ticktext=months_sorted,
        ),
        hovermode='closest',
        legend=dict(orientation='h', yanchor='bottom', y=-0.25),
        template='plotly_white',
        height=500,
        margin=dict(l=60, r=20, t=60, b=80)
    )
    return fig

fig_main = plot_int_step(
    tot_cs_major,
    industry_list=['시중은행','인터넷은행','지방은행','신용카드','캐피탈','저축은행'],
    standard_name='OK',
    start_month=YM,
    show_individual=False,
    title=f'전업권 평균금리 ({year}년 {month}월)'
)
fig_main.show()

### 3.2. cs_viline_graph (Plotly 버전)

원본 matplotlib 클래스의 시각 정체성을 Plotly로 이식:

* **3단 배경**(기준사 / 업권 평균 / 개별사) — alpha 0.40/0.25/0.12
* **신용점수 구간별 짧은 가로선** + YlOrRd 그라데이션 (호버 시 사명/구간/금리 표시)
* **평균금리 회색 라인** + 백분율 라벨 (박스 배경)
* **우측 컬러바** (900점대 → 300점이하)
* 줌·팬·호버 인터랙션 지원

> 외부 모듈 `cs_viline_plotly.py`에 정의된 `plot_cs_viline_plotly` 함수 사용.


In [ ]:
# M5: cs_viline_graph (Plotly 버전)
import sys, os
# 현재 노트북 폴더를 import path에 추가
_this_dir = os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else os.getcwd()
if _this_dir not in sys.path:
    sys.path.insert(0, _this_dir)

from cs_viline_plotly import plot_cs_viline_plotly

fig_cs = plot_cs_viline_plotly(
    tot_cs_major,
    standard_name='OK저축은행',   # 기준사 (사명 정확히 일치해야 함)
    industry_list=['시중은행','인터넷은행','지방은행','신용카드','캐피탈','저축은행'],
    target_month=YM,
    height=720,
)
fig_cs.show()


## 4. HTML 보고서 출력 (단일 파일, 반응형)

Plotly figure를 그대로 임베드한 단일 HTML 파일을 생성. 외부 의존성 없이 브라우저로 더블클릭하여 열 수 있다.

In [ ]:
# Plotly figure → HTML 조각
figs_html = []
figs_html.append(pio.to_html(fig_main, full_html=False, include_plotlyjs='cdn'))
figs_html.append(pio.to_html(fig_cs, full_html=False, include_plotlyjs=False))
# (M5 추가 시 cs_viline 그래프도 여기에 append)

# 단순 통계 요약
summary_rows = (tot_cs_major.groupby('업권', as_index=False)['평균금리']
                  .agg(['count','mean','min','max']))
summary_rows.columns = ['업권','샘플수','평균','최저','최고']
summary_html = (summary_rows.set_index('업권')
                  .map(lambda v: f"{v*100:.2f}%" if isinstance(v,float) and v < 1 else v)
                  .to_html(classes='summary', border=0))

# 반응형 HTML 템플릿 (간단 인라인 CSS)
html = f"""<!DOCTYPE html>
<html lang="ko">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>가계신용대출 금리 트렌드 — {year}년 {month}월</title>
<style>
  body {{ font-family: -apple-system, BlinkMacSystemFont, 'Malgun Gothic', sans-serif;
          max-width: 1200px; margin: 0 auto; padding: 24px; color: #222; }}
  h1 {{ font-size: 1.6rem; border-bottom: 2px solid #333; padding-bottom: 8px; }}
  h2 {{ font-size: 1.2rem; margin-top: 2.5rem; color: #444; }}
  .meta {{ color: #888; font-size: 0.9rem; margin-bottom: 1.5rem; }}
  .summary {{ border-collapse: collapse; width: 100%; font-size: 0.9rem; }}
  .summary th, .summary td {{ padding: 8px 12px; border-bottom: 1px solid #eee; text-align: right; }}
  .summary th {{ background: #f5f5f5; text-align: center; }}
  .summary td:first-child, .summary th:first-child {{ text-align: left; }}
  .chart {{ margin: 1.5rem 0; }}
  .footer {{ margin-top: 4rem; color: #aaa; font-size: 0.8rem; text-align: center; }}
  @media (max-width: 720px) {{
    body {{ padding: 12px; }}
    h1 {{ font-size: 1.3rem; }}
  }}
</style>
</head>
<body>
<h1>가계신용대출 금리 트렌드</h1>
<div class="meta">데이터 기준: {year}년 {month}월 &middot; 생성: {datetime.now():%Y-%m-%d %H:%M}</div>

<h2>1. 업권별 평균금리 요약</h2>
{summary_html}

<h2>2. 평균금리 트렌드</h2>
<div class="chart">{figs_html[0]}</div>
            <h2>3. 신용점수 구간별 금리</h2>
            <div class="chart">{figs_html[1]}</div>

<div class="footer">© 가계신용대출 금리 트렌드 통합 노트북 (미리보기 빌드)</div>
</body>
</html>"""

out_path = f"{OUT_REPORT}/report_{YM}.html"
with open(out_path, 'w', encoding='utf-8') as f:
    f.write(html)
print(f"보고서 생성: {out_path}")
print(f"브라우저에서 열기:  file://{os.path.abspath(out_path)}")

---

## 다음 단계 (본 단계로 진입 시)

1. **M1 보강** — 다월 백필. 본 노트북을 함수화하여 과거 24개월 누적 수집 → tot_cs를 시계열로 풍부화.
2. **M2 신규 크롤링 8~10개** — 잔액(예금보험공사, 협회별), 중금리 취급실적, 금리대별 취급비중.
3. **M5 cs_viline Plotly** — 가장 어려운 부분. matplotlib 코드를 1:1로 이식하지 말고 Plotly 친화적 디자인으로 재해석 권장.
4. **반응형 향상** — 현재는 단순 미디어쿼리만 적용. Tailwind CSS 또는 grid 레이아웃 도입 시 모바일에서 차트 가독성 개선.
5. **배포** — `report_{YM}.html` 단일 파일이라 GitHub Pages 또는 Netlify에 그대로 push 가능.